# Notebook 05: Feature Engineering

## I. Giới thiệu
Mục tiêu: Xây dựng và đánh giá các đặc trưng (Feature) nhằm nâng cao chất lượng mô hình dự báo nhiệt độ.


## II. Đọc dữ liệu


In [ ]:
import pandas as pd
import numpy as np

city_df = pd.read_csv('../data/processed/tokyo_cleaned.csv', index_col=0, parse_dates=True)


## III. Ý tưởng xây dựng Feature
Từ kết quả EDA:
- **Tính mùa vụ**: Cần đặc trưng Tháng (Month), Năm (Year).
- **Chu kỳ**: Nhiệt độ tháng này phụ thuộc lớn vào tháng trước (Lag 1) và cùng kỳ năm ngoái (Lag 12).
- **Xu hướng chung**: Nhiệt độ trung bình trượt 12 tháng (Rolling Mean 12).


## IV. Xây dựng Feature
### 1. Time Features


In [ ]:
city_df['Year'] = city_df.index.year
city_df['Month'] = city_df.index.month


### 2. Lag & Rolling Features


In [ ]:
city_df['Lag_1'] = city_df['AverageTemperature'].shift(1)
city_df['Lag_12'] = city_df['AverageTemperature'].shift(12)
city_df['Rolling_Mean_12'] = city_df['AverageTemperature'].rolling(window=12).mean()

# Bỏ các dòng bị NaN do tạo Lag
city_df = city_df.dropna()
print(city_df[['AverageTemperature', 'Lag_1', 'Lag_12', 'Rolling_Mean_12']].head())


## V. Đánh giá và Lựa chọn Feature
Sử dụng Correlation Matrix để xem sự tương quan giữa các đặc trưng mới và mục tiêu.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr = city_df[['AverageTemperature', 'Year', 'Month', 'Lag_1', 'Lag_12', 'Rolling_Mean_12']].corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Ma trận Tương quan')
plt.show()


## VI. Lưu dữ liệu mới
Lưu bộ dữ liệu hoàn chỉnh để phục vụ khâu Train mô hình ở Notebook 06.


In [ ]:
# Lưu vào PostgreSQL
# city_df.to_sql('tokyo_features', engine, if_exists='replace')

city_df.to_csv('../data/processed/tokyo_features.csv')


## VII. Kết luận
Bộ dữ liệu đã được làm giàu bởi các Lag feature và Rolling feature. Điều này sẽ giúp các mô hình Machine Learning nắm bắt tốt xu hướng chuỗi thời gian.
